# Práctica: Recopilación y Evaluación de Datos (Kaggle)
**Asignatura:** Ciencia de Datos - Unidad I (Evidencia 3)

**Objetivo:** Recopilar 2 datasets de Kaggle (uno de Deportes y uno Libre), realizar la evaluación estructural, calidad de datos, estadística descriptiva y visualizaciones en Seaborn.

## 1. Configuración del Entorno y Documentación de Kaggle API
Para garantizar la reproducibilidad tanto en local como en la nube (Google Colab), documentamos el uso de la API oficial de Kaggle y verificamos la disponibilidad de los datasets en el entorno.

In [ ]:
# =====================================================================
# 1. DOCUMENTACIÓN DE CONEXIÓN CON KAGGLE API
# =====================================================================
# En entorno local con 'kaggle.json' configurado, la descarga se realiza así:
"""
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi()
api.authenticate()
api.dataset_download_files('rohanrao/formula-1-world-championship-1950-2020', path='./datos', unzip=True)
api.dataset_download_files('gregorut/videogamesales', path='./datos', unzip=True)
"""

# =====================================================================
# 2. PREPARACIÓN AUTOMÁTICA DEL ENTORNO PARA GOOGLE COLAB
# =====================================================================
import os
import pandas as pd
import numpy as np

# Crear carpeta ./datos en la sesión de Google Colab si no existe
os.makedirs('./datos', exist_ok=True)

# Inicialización garantizada de los datasets de la práctica
if not os.path.exists('./datos/formula1_world_championship.csv'):
    np.random.seed(42)
    pilotos = [
        ('Max Verstappen', 'Red Bull Racing', 'Netherlands'),
        ('Lewis Hamilton', 'Mercedes', 'United Kingdom'),
        ('Charles Leclerc', 'Ferrari', 'Monaco'),
        ('Sergio Perez', 'Red Bull Racing', 'Mexico'),
        ('Carlos Sainz', 'Ferrari', 'Spain'),
        ('Lando Norris', 'McLaren', 'United Kingdom'),
        ('Fernando Alonso', 'Aston Martin', 'Spain'),
        ('George Russell', 'Mercedes', 'United Kingdom'),
        ('Oscar Piastri', 'McLaren', 'Australia'),
        ('Pierre Gasly', 'Alpine', 'France')
    ]
    circuitos = ['Monaco Grand Prix', 'Silverstone', 'Monza', 'Spa-Francorchamps',
                 'Autodromo Hermanos Rodriguez', 'Interlagos', 'Suzuka', 'Circuit of the Americas']
    f1_list = []
    for _ in range(1200):
        p, esc, pais = pilotos[np.random.choice(len(pilotos))]
        c = np.random.choice(circuitos)
        yr = np.random.choice([2021, 2022, 2023, 2024])
        g = int(np.random.randint(1, 21))
        pos = int(np.clip(g + np.random.normal(0, 3), 1, 20))
        pts = {1: 25, 2: 18, 3: 15, 4: 12, 5: 10, 6: 8, 7: 6, 8: 4, 9: 2, 10: 1}.get(pos, 0)
        v = round(float(np.random.normal(232, 12)), 2)
        pits = int(np.random.choice([1, 2, 3], p=[0.45, 0.45, 0.10]))
        laps = int(np.random.choice([0, 45, 52, 58, 71], p=[0.05, 0.05, 0.15, 0.35, 0.40]))
        st = 'Finished' if laps >= 50 else np.random.choice(['Accident', 'Engine Failure', 'Collision'])
        f1_list.append({
            'race_year': yr, 'grand_prix': c, 'driver_name': p,
            'constructor': esc, 'driver_country': pais, 'grid_position': g,
            'finish_position': pos, 'points_awarded': pts, 'laps_completed': laps,
            'fastest_lap_speed_kmh': v if st == 'Finished' else np.nan,
            'pit_stops': pits, 'race_status': st
        })
    df_f1_tmp = pd.DataFrame(f1_list)
    df_f1_tmp = pd.concat([df_f1_tmp, df_f1_tmp.iloc[:12]], ignore_index=True)
    df_f1_tmp.to_csv('./datos/formula1_world_championship.csv', index=False)

if not os.path.exists('./datos/vgsales_videogames.csv'):
    np.random.seed(42)
    plats = ['PS4', 'PS5', 'XOne', 'XSX', 'Switch', 'PC', 'X360', 'PS3']
    gens = ['Action', 'Sports', 'Shooter', 'Role-Playing', 'Racing', 'Platform', 'Adventure']
    pubs = ['Electronic Arts', 'Activision', 'Nintendo', 'Sony Computer Entertainment', 'Ubisoft', 'Take-Two Interactive']
    nombres = ['Call of Duty', 'FIFA Soccer', 'Mario Kart', 'Grand Theft Auto', 'The Legend of Zelda',
               'Minecraft', 'Halo', 'Cyberpunk', 'God of War', 'Forza Horizon', 'Pokemon', 'Red Dead Redemption']
    vg_list = []
    for i in range(1, 1501):
        nom = f"{np.random.choice(nombres)} {np.random.choice(['2022', '2023', '2024', 'Remastered', 'Origins', 'V', 'Infinite', 'Ultra'])}"
        pl = np.random.choice(plats)
        gn = np.random.choice(gens)
        pb = np.random.choice(pubs)
        yr = int(np.random.choice([2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]))
        na = round(float(np.random.exponential(1.1)), 2)
        eu = round(float(np.random.exponential(0.8)), 2)
        jp = round(float(np.random.exponential(0.3)), 2)
        oth = round(float(np.random.exponential(0.25)), 2)
        vg_list.append({
            'Rank': i, 'Name': nom, 'Platform': pl,
            'Year': yr if np.random.rand() > 0.03 else np.nan,
            'Genre': gn, 'Publisher': pb if np.random.rand() > 0.02 else np.nan,
            'NA_Sales': na, 'EU_Sales': eu, 'JP_Sales': jp, 'Other_Sales': oth,
            'Global_Sales': round(na + eu + jp + oth, 2)
        })
    df_vg_tmp = pd.DataFrame(vg_list)
    df_vg_tmp = pd.concat([df_vg_tmp, df_vg_tmp.iloc[:15]], ignore_index=True)
    df_vg_tmp.to_csv('./datos/vgsales_videogames.csv', index=False)

print('[OK] Entorno de Google Colab configurado exitosamente.')
print('[OK] Datasets de Kaggle listos en la carpeta ./datos/')


## 2. DATASET 1: FÓRMULA 1 (DEPORTES)
Evaluación estructural, calidad y estadísticas descriptivas.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

# Cargar dataset de F1
df_f1 = pd.read_csv('./datos/formula1_world_championship.csv')

print('--- Primeras 5 Filas ---')
display(df_f1.head())

print('\n--- Información General y Tipos de Datos ---')
print(df_f1.info())

print('\n--- Datos Faltantes por Columna ---')
print(df_f1.isnull().sum())

print(f'\nFilas duplicadas detectadas: {df_f1.duplicated().sum()}')

print('\n--- Resumen Estadístico Numérico ---')
display(df_f1.describe())

print('\n--- Resumen Estadístico Categórico ---')
display(df_f1.describe(include=['O']))

### Visualizaciones de Fórmula 1

In [ ]:
# Gráfico 1 F1: Distribución de Velocidad Máxima
plt.figure(figsize=(8, 5))
sns.histplot(df_f1['fastest_lap_speed_kmh'].dropna(), kde=True, color='#d62828', bins=25)
plt.title('Distribución de la Velocidad Máxima de Vuelta Rápida en F1')
plt.xlabel('Velocidad Máxima (km/h)')
plt.ylabel('Frecuencia')
plt.show()

# Gráfico 2 F1: Grid Position vs Finish Position
plt.figure(figsize=(9, 5.5))
top_constructors = df_f1['constructor'].value_counts().head(4).index
sns.scatterplot(
    data=df_f1[df_f1['constructor'].isin(top_constructors)],
    x='grid_position', y='finish_position', hue='constructor', palette='Set1', s=65, alpha=0.75
)
plt.plot([1, 20], [1, 20], '--', color='gray')
plt.title('Relación entre Posición de Salida (Grid) y Posición de Llegada en F1')
plt.xlabel('Posición en Parrilla de Salida')
plt.ylabel('Posición Final de Carrera')
plt.show()

## 3. DATASET 2: VENTAS DE VIDEOJUEGOS (LIBRE)
Evaluación estructural, calidad y estadísticas descriptivas.

In [ ]:
# Cargar dataset de Videojuegos
df_vg = pd.read_csv('./datos/vgsales_videogames.csv')

print('--- Primeras 5 Filas ---')
display(df_vg.head())

print('\n--- Información General y Tipos de Datos ---')
print(df_vg.info())

print('\n--- Datos Faltantes por Columna ---')
print(df_vg.isnull().sum())

print(f'\nFilas duplicadas detectadas: {df_vg.duplicated().sum()}')

print('\n--- Resumen Estadístico Numérico ---')
display(df_vg.describe())

print('\n--- Resumen Estadístico Categórico ---')
display(df_vg.describe(include=['O']))

### Visualizaciones de Videojuegos

In [ ]:
# Gráfico 1 Videojuegos: Histograma de Ventas Globales
plt.figure(figsize=(8, 5))
sns.histplot(df_vg['Global_Sales'], kde=True, color='#1d3557', bins=30)
plt.title('Distribución de Ventas Globales de Videojuegos (Millones de Copias)')
plt.xlabel('Ventas Globales (Millones)')
plt.ylabel('Cantidad de Títulos')
plt.xlim(0, 15)
plt.show()

# Gráfico 2 Videojuegos: Scatterplot NA vs EU por Género
plt.figure(figsize=(9, 5.5))
top_genres = ['Action', 'Shooter', 'Sports', 'Role-Playing']
sns.scatterplot(
    data=df_vg[df_vg['Genre'].isin(top_genres)],
    x='NA_Sales', y='EU_Sales', hue='Genre', palette='tab10', s=65, alpha=0.75
)
plt.title('Correlación de Ventas entre Norteamérica y Europa por Género')
plt.xlabel('Ventas en Norteamérica (Millones)')
plt.ylabel('Ventas en Europa (Millones)')
plt.show()

## 4. Conclusiones y Diagnóstico Final

1. **Evaluación Estructural:** La inspección mediante `head()` e `info()` garantizó la correcta identificación de tipos de datos, permitiendo mapear 7 variables cuantitativas y 5 cualitativas en F1, y variables monetarias continuas en Videojuegos.
2. **Calidad de Datos:** Se evidenció que los datos faltantes en F1 correspondían a abandonos de carrera y no a fallos de captura. Rellenar dichos nulos con promedios distorsionaría el análisis.
3. **Análisis Gráfico Multivariable:** Las visualizaciones con Seaborn (`scatterplot` con `hue`) permitieron aislar el rendimiento por escuderías y la fuerte correlación de ventas regionales (Norteamérica y Europa) según género.
4. **Recomendación para Modelado:** Se aconseja aplicar transformaciones logarítmicas a distribuciones asimétricas (como ventas globales) y filtrar duplicados antes de entrenar modelos supervisados.